# 01 — Exploratory data analysis

**Goal:** understand `yield_df.csv` well enough to make safe decisions for the one-year-ahead forecast in M2.

Each section answers one question from `PLAN.md` (M1). The decisions are collected at the end.

Data: Kaggle `patelris/crop-yield-prediction-dataset` (FAO yield and pesticides, World Bank rainfall and temperature). Run `make data` first.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.width", 120)

RAW = Path("../data/raw")
KEY = ["Area", "Item", "Year"]

raw = pd.read_csv(RAW / "yield_df.csv", index_col=0).rename(
    columns={
        "hg/ha_yield": "yield_hg_ha",
        "average_rain_fall_mm_per_year": "rain_mm",
        "pesticides_tonnes": "pesticides_t",
        "avg_temp": "temp_c",
    }
)
print(raw.shape)
raw.head()

(28242, 7)


,Area,Item,Year,yield_hg_ha,rain_mm,pesticides_t,temp_c
0,Albania,Maize,1990,36613,1485.0,121.0,16.37
1,Albania,Potatoes,1990,66667,1485.0,121.0,16.37
2,Albania,"Rice, paddy",1990,23333,1485.0,121.0,16.37
3,Albania,Sorghum,1990,12500,1485.0,121.0,16.37
4,Albania,Soybeans,1990,7000,1485.0,121.0,16.37


## 1. Duplicates

There should be exactly one row per (Area, Item, Year). Is there?

In [2]:
n_rows = len(raw)
n_keys = raw[KEY].drop_duplicates().shape[0]
print(f"rows: {n_rows:,}   unique keys: {n_keys:,}   extra rows: {n_rows - n_keys:,}")
print(f"exact full-row duplicates: {raw.duplicated().sum():,}")

dup = raw[raw.duplicated(KEY, keep=False)]
print("\nWhich columns vary within a duplicated key?")
print(dup.groupby(KEY)[["yield_hg_ha", "rain_mm", "pesticides_t", "temp_c"]].nunique().gt(1).sum())
print("\nCopies per key, by country (top 10):")
print(dup.groupby(KEY).size().groupby(level="Area").max().sort_values(ascending=False).head(10))

rows: 28,242   unique keys: 13,130   extra rows: 15,112
exact full-row duplicates: 2,310

Which columns vary within a duplicated key?
yield_hg_ha        0
rain_mm            0
pesticides_t       0
temp_c          3690
dtype: int64

Copies per key, by country (top 10):
Area
India             22
Brazil            11
Pakistan           9
Mexico             8
Canada             6
Japan              6
Australia          6
Indonesia          6
United Kingdom     5
Turkey             5
dtype: int64


In [3]:
# Where does it come from? temp.csv has several readings per country-year (different stations/cities).
temp = pd.read_csv(RAW / "temp.csv")
print(temp.groupby(["country", "year"]).size().value_counts().sort_index().head(8))
temp.query("country == 'India' and year == 2000").avg_temp.describe()

1    20520
2     3451
3     1192
4      197
5      791
6      742
7      191
8      190
Name: count, dtype: int64


count    22.000000
mean     25.991364
std       0.849074
min      24.570000
25%      25.407500
50%      26.015000
75%      26.475000
max      27.440000
Name: avg_temp, dtype: float64

**Finding.** More than half the rows (15,112 of 28,242) are duplicate keys. Yield, rainfall and pesticides are identical within a key; **only temperature differs**. The cause is `temp.csv`, which has up to 22 readings per country-year (several weather stations), and the original merge made one row per reading.

**Resolution.** Collapse to one row per key: take yield, rain and pesticides as they are, and **average temperature** across stations. That leaves 13,130 rows. Without this step, a random split would put copies of the same (country, crop, year) on both sides, which would hand the model the answer.

In [4]:
panel = raw.groupby(KEY, as_index=False).agg(
    yield_hg_ha=("yield_hg_ha", "first"),
    rain_mm=("rain_mm", "first"),
    pesticides_t=("pesticides_t", "first"),
    temp_c=("temp_c", "mean"),
)
assert not panel.duplicated(KEY).any()
panel["log_yield"] = np.log(panel.yield_hg_ha)
panel.shape

(13130, 8)

## 2. Coverage

Which (country, crop) series have a full 1990–2013 history, and where are the gaps?

In [5]:
print("years present:", sorted(panel.Year.unique()))
print("missing years:", sorted(set(range(1990, 2014)) - set(panel.Year)))
rain_raw = pd.read_csv(RAW / "rainfall.csv")
rain_raw.columns = rain_raw.columns.str.strip()
print("rainfall.csv rows for 2003:", (rain_raw.Year == 2003).sum())

years present: [np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013)]
missing years: [2003]
rainfall.csv rows for 2003: 0


In [6]:
yield_raw = pd.read_csv(RAW / "yield.csv")[["Area", "Item", "Year", "Value"]]
pairs = panel[["Area", "Item"]].drop_duplicates()
has_2003 = pairs.merge(yield_raw.query("Year == 2003"), on=["Area", "Item"], how="left").Value.notna()
print(f"series with a 2003 yield in yield.csv: {has_2003.sum()} / {len(pairs)}")

check = panel.merge(yield_raw, on=KEY, how="left")
print("yield_df matches yield.csv on every row:", (check.yield_hg_ha == check.Value).all())

series with a 2003 yield in yield.csv: 575 / 598
yield_df matches yield.csv on every row: True


In [7]:
hist = panel.groupby(["Area", "Item"]).Year.agg(n_years="count", first="min", last="max")
print(f"{len(hist)} series; {(hist.n_years == 23).sum()} have all 23 available years (1990–2013 minus 2003)")
print(hist.n_years.value_counts().sort_index())

598 series; 507 have all 23 available years (1990–2013 minus 2003)
n_years
1       1
2       7
3       1
4       4
6       1
8       3
9       3
10      3
12      1
13      6
16      2
17      1
18      1
20      8
21     47
22      2
23    507
Name: count, dtype: int64


In [8]:
cov = panel.pivot_table(index="Item", columns="Year", values="Area", aggfunc="nunique")
px.imshow(cov, aspect="auto", color_continuous_scale="Greens",
          title="Number of countries reporting each crop, per year").show()

In [9]:
# How many rows are left once we need 1, 2 or 3 *consecutive* prior years (lags must be exactly t-1, t-2, t-3)?
years = set(zip(panel.Area, panel.Item, panel.Year))
def has_lags(r, k):
    return all((r.Area, r.Item, r.Year - j) in years for j in range(1, k + 1))
for k in (1, 2, 3):
    ok = panel.apply(has_lags, axis=1, k=k)
    print(f"lags 1..{k}: {ok.sum():>6,} usable rows ({ok.mean():.0%})   test-period rows (2009–13): {ok[panel.Year >= 2009].sum():,}")

lags 1..1: 11,951 usable rows (91%)   test-period rows (2009–13): 2,926


lags 1..2: 10,777 usable rows (82%)   test-period rows (2009–13): 2,914


lags 1..3:  9,611 usable rows (73%)   test-period rows (2009–13): 2,910


**Findings.**
- **2003 is missing for every series.** `rainfall.csv` has no 2003 rows, so the original inner merge dropped that year entirely. Yields for 2003 do exist in `yield.csv` for 575 of the 598 series, and `yield_df` matches `yield.csv` exactly on every row.
- 598 (country, crop) series. **507 are complete** (all 23 available years). A tail of about 20 series has fewer than 10 years, and 6 have gaps besides 2003.
- Lags must mean "exactly one, two or three years earlier", not "the previous row". If they are computed by position, the 2004 row's "lag 1" silently becomes 2002.
- **Minimum history:** lags 1–3 need three consecutive prior years, so the first usable target year is 1993. Without 2003, the target years 2004–2006 also lose rows. Recovering 2003 from `yield.csv` fixes that.

## 3. Scale

How does yield range across crops?

In [10]:
panel.groupby("Item").yield_hg_ha.describe()[["min", "25%", "50%", "75%", "max"]].astype(int).sort_values("50%")

,min,25%,50%,75%,max
Item,,,,,
Sorghum,578,7192,12333,24126,206000
Soybeans,50,10000,16204,22213,41609
Wheat,1706,15890,24318,41287,99387
Maize,849,14068,24489,53185,207556
"Rice, paddy",2034,22870,33553,48932,103895
Sweet potatoes,8799,54089,87500,149969,400000
Yams,11475,62844,92403,132997,250000
Plantains and others,21350,58837,97024,125000,418505
Cassava,11778,58596,100000,132173,385818


In [11]:
fig = px.box(panel, x="Item", y="yield_hg_ha", log_y=True, points=False,
             title="Yield by crop (hg/ha, log scale)")
fig.update_xaxes(categoryorder="median ascending").show()

In [12]:
print("skew, raw yield:", round(panel.yield_hg_ha.skew(), 2),
      "  skew, log yield:", round(panel.log_yield.skew(), 2))

skew, raw yield: 2.08   skew, log yield: -0.22


**Finding.** Medians range from about 12k hg/ha (sorghum) to 162k (potatoes). Across all rows, the range runs from 50 (Tajikistan soybeans) to 500k. Tubers (potatoes, cassava, yams, sweet potatoes) sit roughly 5× above the cereals. Raw yield is heavily right-skewed; log yield is close to symmetric.

**Decision.** Use a **log target** with one global model, and pass `Item` as a categorical feature. With the log, an error means a percentage, which puts potatoes and soybeans on the same footing, and one model can share the trend signal across crops. Training one model per crop would leave Yams and Plantains with fewer than 1,000 rows each.

## 4. Feature quality

Do rainfall, pesticides and temperature carry year-to-year signal?

In [13]:
per_country = panel.groupby("Area").agg(
    rain_values=("rain_mm", "nunique"),
    pest_values=("pesticides_t", "nunique"),
    temp_values=("temp_c", "nunique"),
)
print("countries:", len(per_country))
print("  with ONE rainfall value across all years:", (per_country.rain_values == 1).sum())
print("  with ONE pesticides value across all years:", (per_country.pest_values == 1).sum())
print("  with ONE temperature value across all years:", (per_country.temp_values == 1).sum())

countries: 101
  with ONE rainfall value across all years: 101
  with ONE pesticides value across all years: 1
  with ONE temperature value across all years: 0


In [14]:
panel[["log_yield", "rain_mm", "pesticides_t", "temp_c"]].corr(method="spearman")["log_yield"].round(3)

log_yield       1.000
rain_mm         0.075
pesticides_t    0.220
temp_c         -0.106
Name: log_yield, dtype: float64

In [15]:
# Within-series: after removing each series' own level, does this year's weather move yield?
w = panel.copy()
for c in ["log_yield", "temp_c", "pesticides_t"]:
    w[c + "_dm"] = w[c] - w.groupby(["Area", "Item"])[c].transform("mean")
w[["log_yield_dm", "temp_c_dm", "pesticides_t_dm"]].corr(method="spearman")["log_yield_dm"].round(3)

log_yield_dm       1.000
temp_c_dm          0.144
pesticides_t_dm    0.169
Name: log_yield_dm, dtype: float64

In [16]:
px.scatter(panel, x="pesticides_t", y="yield_hg_ha", color="Item", log_x=True, log_y=True,
           opacity=0.4, title="Yield vs pesticides (country totals, log–log)").show()

**Findings.**
- **Rainfall is a constant per country.** Every one of the 101 countries has a single value repeated across all years. It's a long-run climate average, not weather. It carries no year-to-year information and duplicates `Area`, so it adds nothing to a model that already knows the country.
- **Pesticides** vary over time, but they are a **country total across all crops** (tonnes, not per hectare), so they mostly measure country size and how intensive its farming is. The overall correlation with log yield is weak and positive (Spearman ≈ 0.22).
- **Temperature** is the station-averaged annual mean. Its overall correlation is weakly negative (≈ −0.11), which mostly reflects hot countries being poorer. Within a series (after removing each series' mean), the correlations are weak and positive (temperature ≈ 0.14, pesticides ≈ 0.17). That's consistent with all three drifting upward over time, so it isn't evidence that they drive yield.

**Decision.** Drop rainfall as a feature (it's covered by `Area`). Keep prior-year pesticides and temperature as cheap features and let the model decide. **Leave out same-year weather:** the signal is weak, and using it would assume we know this year's weather before the harvest.

## 5. Trend

How strong is the upward trend, and how well does last year's yield alone predict this year's?

In [17]:
idx = (panel.assign(base=lambda d: d.groupby(["Area", "Item"]).log_yield.transform("first"))
            .assign(rel=lambda d: np.exp(d.log_yield - d.base))
            .groupby(["Item", "Year"]).rel.median().reset_index())
px.line(idx, x="Year", y="rel", color="Item",
        title="Median yield relative to each series' first year (1.0 = first year)").show()

In [18]:
p = panel.sort_values(KEY).copy()
g = p.groupby(["Area", "Item"])
p["prev_year"] = g.Year.shift(1)
p["prev_yield"] = g.yield_hg_ha.shift(1)
nv = p[p.Year - p.prev_year == 1].copy()          # true one-year-apart pairs only
nv["ape"] = (nv.yield_hg_ha - nv.prev_yield).abs() / nv.yield_hg_ha
nv["growth"] = np.log(nv.yield_hg_ha / nv.prev_yield)

print(f"corr(log yield, log last-year yield): {np.corrcoef(nv.log_yield, np.log(nv.prev_yield))[0, 1]:.3f}")
print(f"naive 'last year' MAPE — all years: {nv.ape.mean():.1%}   test period 2009–13: {nv.loc[nv.Year >= 2009, 'ape'].mean():.1%}")
print(f"median APE: {nv.ape.median():.1%}")
nv.groupby("Item").agg(naive_MAPE=("ape", "mean"), median_yearly_growth=("growth", "median")).sort_values("naive_MAPE").round(3)

corr(log yield, log last-year yield): 0.979
naive 'last year' MAPE — all years: 13.2%   test period 2009–13: 12.1%
median APE: 6.0%


,naive_MAPE,median_yearly_growth
Item,,
Plantains and others,0.076,0.001
Cassava,0.086,0.001
Sweet potatoes,0.089,0.002
Yams,0.095,0.002
Potatoes,0.095,0.008
"Rice, paddy",0.106,0.011
Wheat,0.143,0.002
Soybeans,0.162,0.003
Maize,0.169,0.014


In [19]:
nv.loc[nv.growth.abs().nlargest(8).index, ["Area", "Item", "Year", "prev_yield", "yield_hg_ha", "growth"]]

,Area,Item,Year,prev_yield,yield_hg_ha,growth
11841,Tajikistan,Soybeans,2009,112.0,2459,3.089011
11840,Tajikistan,Soybeans,2008,824.0,112,-1.995672
1376,Botswana,Sorghum,2001,1573.0,11538,1.992661
8409,Morocco,Maize,1996,1303.0,9274,1.962545
8478,Morocco,Sorghum,1996,866.0,6055,1.944755
3907,Eritrea,Wheat,2002,11658.0,1706,-1.921841
6634,Kazakhstan,Sorghum,1999,7692.0,50000,1.871842
8418,Morocco,Maize,2006,2036.0,12198,1.790285


**Findings.**
- The trend is real but slow. Median yearly growth is about **+1.4% for maize** and **+1.1% for rice**, and under 1% for most other crops.
- **Last year's yield is a very strong predictor.** The log correlation is about 0.98, and the naive MAPE is about **13% overall (12% in 2009–13)**, with a median error around 6%. Year-to-year noise matters much more than trend, and the mean MAPE is pulled up by a few huge jumps (e.g. Tajikistan soybeans rose 22× in 2009, the same series fell 86% in 2008). These look like reporting artefacts on tiny harvested areas.
- **Implication for M2:** the baseline will be hard to beat. Useful signals include multi-year averages (lags 1–3 smooth out noise) and a trend term. MAPE is fragile when yields are tiny, so it helps to report the median APE or RMSE on log yield alongside it.

## 6. Showcase: Thailand, rice (paddy)

In [20]:
th = panel.query("Area == 'Thailand' and Item == 'Rice, paddy'")[["Year", "yield_hg_ha"]]
th_2003 = yield_raw.query("Area == 'Thailand' and Item == 'Rice, paddy' and Year == 2003")
fig = px.line(th, x="Year", y="yield_hg_ha", markers=True,
              title="Thailand — Rice, paddy yield (hg/ha); 2009–13 is the test period")
fig.add_scatter(x=th_2003.Year, y=th_2003.Value, mode="markers", marker_symbol="x", marker_size=10,
                name="2003 (from yield.csv)")
fig.add_vrect(x0=2008.5, x1=2013.5, fillcolor="grey", opacity=0.15, line_width=0)
fig.show()
growth = np.log(th.yield_hg_ha.iloc[-1] / th.yield_hg_ha.iloc[0]) / (th.Year.iloc[-1] - th.Year.iloc[0])
print(f"1990: {th.yield_hg_ha.iloc[0]:,}  2013: {th.yield_hg_ha.iloc[-1]:,}  average growth ≈ {growth:.1%}/yr")

1990: 19,556  2013: 31,463  average growth ≈ 2.1%/yr


Thailand rice rose from about 19.6k to about 31.5k hg/ha between 1990 and 2013 (about 2% a year), and it plateaus in 2011–13. It's a clean, complete series, which makes it a good default for the app. The plateau at the end of the series is exactly where "last year's value" is hard to beat.

## 7. Decisions for M2

| # | Topic | Decision |
|---|---|---|
| 1 | Duplicates | Build the panel with one row per (Area, Item, Year): yield, rain and pesticides as they are, **temperature averaged** across stations. Test: no duplicate keys. |
| 2 | Missing 2003 | Build yields from `yield.csv` (1990–2013) so 2003 is present for lags. Take features from the other raw files. `yield_df` values match `yield.csv` exactly, so nothing else changes. |
| 3 | Lags | Compute lags by **calendar year** (merge on `Year - k`), not by row position. A missing year gives NaN, never an older value. Test: lags never read rows with `Year >= t`. |
| 4 | Min history | Lags 1–3. Keep rows with at least lag 1 and let LightGBM handle NaNs in lags 2–3. The baseline needs lag 1 exactly. Evaluate the baseline and the model **on the same rows**. |
| 5 | Target | `log(yield)`, one global LightGBM model, with `Item` and `Area` as categoricals. Report MAPE (plus median APE) on yield and RMSE on log yield, overall and per crop. |
| 6 | Rainfall | **Drop it.** It's a constant per country and adds nothing beyond `Area`. |
| 7 | Weather / pesticides | Use **prior-year** temperature and pesticides only. No same-year weather. Pesticides are country totals, a proxy for farming intensity. |
| 8 | Trend | Add `Year` and a lag-based trend feature (e.g. lag1 − lag3). The trend is small (≈ 0–1.5% a year), so noise dominates. |
| 9 | Split | Train on target years ≤ 2008, test on 2009–2013. Random 80/20 only for the leakage demo. |
| 10 | Expectations | Naive MAPE is about 12% on the test period, with a log correlation of about 0.98. Beating it by a clear margin isn't guaranteed; report honestly either way. |

### Open questions
- Should the few extreme jumps (e.g. Tajikistan soybeans) be clipped or kept? For now: keep them, and report median APE next to MAPE so they don't dominate.
- The 23 series without a 2003 yield in `yield.csv` keep a gap there. That's fine with calendar-year lags.